<a href="https://colab.research.google.com/github/Miguithub/NPP_Cripto_D-W/blob/ID/ID_(Ingenieria_de_Datos)07_2023_07_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ingeniería de datos diaria · original + reducido
Se conservan las 13 familias y sus fórmulas por activo. Se eliminan los cortes verticales, las aperturas repetidas y la limpieza de cinco minutos. Las series temporales se calculan por identidad antes de asignarlas al ranking causal de tres días.

Entrada: el parquet diario definido en `ORIGINAL`. El reducido se regenera con fechas desde esa fuente. Los archivos originales no se sobrescriben. Las variables están disponibles al cierre de `bar_end`; los objetivos corresponden a cierres posteriores.

Las ventanas diarias propuestas (3, 7, 14 y 30 días) son parámetros iniciales, no equivalencias de los intervalos de cinco minutos ni valores optimizados. Deben fijarse con entrenamiento/validación. El universo debe estar definido sin selección retrospectiva por supervivencia; el esquema por sí solo no verifica esa condición.

In [ ]:
from google.colab import drive
from pathlib import Path
from dataclasses import dataclass, replace, asdict
import json
import numpy as np
import pandas as pd

drive.mount('/content/drive')
RUTA = Path('/content/drive/MyDrive/Neural/NPP/Cripto/Crypto Dayli')
ORIGINAL = RUTA / '062023-062026.parquet'
REDUCIDO = RUTA / '062023-062026_red.parquet'
SALIDA = RUTA / '062023-062026_red_features_diarias.parquet'
VENTANA_RANKING = 3
HORIZONTES = (1, 2, 3, 7)

Mounted at /content/drive


Configura el archivo original y las salidas. La celda de carga regenerará REDUCIDO con fechas en un archivo nuevo. Los horizontes representan días y los objetivos se guardan separados de las características.

In [ ]:
@dataclass(frozen=True)
class FeatureConfig:
    time_col: str = 'bar_end'
    ys_col: str = 'YS'
    price_suffix: str = 'USDT'
    volume_suffix: str = '_v'
    trades_suffix: str = '_activity_trades_proxy'
    range_suffix: str = '_range_rel_proxy'
    illiq_suffix: str = '_illiquidity_proxy'
    active_suffix: str = '_active_trading'
    observed_suffix: str = '_observed'
    buy_suffix: str = '_taker_buy_usdt_proxy'
    sell_suffix: str = '_taker_sell_usdt_proxy'
    yp_suffix: str = '_yp'
    primary_window: int = 30
    min_periods: int = 10
    ewm_span: int = 7
    multiscale_windows: tuple[int, ...] = (3, 7, 14, 30)
    return_lag: int = 1
    stress_quantile: float = 0.9
    recovery_tolerance: float = 0.15
    recovery_hold_bars: int = 3
    recovery_event_ewm_alpha: float = 0.2
    eps: float = 1e-12
    dtype: str = 'float32'
    asset_block_size: int = 32

def chunks(seq, size):
    for start in range(0, len(seq), size):
        yield list(seq[start:start + size])

def base_col(asset, kind, cfg):
    if kind == 'P':
        return f'{asset}{cfg.price_suffix}'
    suffix = {'V': cfg.volume_suffix, 'C': cfg.trades_suffix, 'R': cfg.range_suffix, 'L': cfg.illiq_suffix, 'A': cfg.active_suffix, 'O': cfg.observed_suffix, 'B': cfg.buy_suffix, 'S': cfg.sell_suffix, 'yp': cfg.yp_suffix}[kind]
    return f'{asset}{suffix}'

def feat_col(asset, feature):
    return f'{asset}__{feature}'

def load_base(df, assets, kind, cfg):
    cols = [base_col(a, kind, cfg) for a in assets]
    out = df.loc[:, cols].astype('float64', copy=False)
    out.columns = assets
    return out

def load_feature(df, assets, feature):
    cols = [feat_col(a, feature) for a in assets]
    out = df.loc[:, cols].astype('float64', copy=False)
    out.columns = assets
    return out

def name_block(x, feature, cfg):
    out = x.astype(cfg.dtype, copy=False)
    out.columns = [feat_col(a, feature) for a in out.columns]
    return out

def merge_blocks(df, blocks):
    """
    Añade todas las features de una celda en un único concat.
    Permite volver a ejecutar una celda sin duplicar columnas.
    """
    block = pd.concat(blocks, axis=1, copy=False)
    overlap = df.columns.intersection(block.columns)
    if len(overlap):
        df = df.drop(columns=list(overlap))
    return pd.concat([df, block], axis=1, copy=False)

def robust_z_causal(x, window, min_periods, eps):
    """
    Compara x_t exclusivamente contra historia t-1,...,t-W.
    """
    hist = x.shift(1)
    roll = hist.rolling(window=window, min_periods=min_periods)
    median = roll.median()
    q25 = roll.quantile(0.25)
    q75 = roll.quantile(0.75)
    scale = (q75 - q25) / 1.349
    return (x - median) / (scale + eps)

def rolling_robust_dispersion(x, window, min_periods, eps):
    roll = x.rolling(window=window, min_periods=min_periods)
    q25 = roll.quantile(0.25)
    q75 = roll.quantile(0.75)
    return (q75 - q25) / 1.349 + eps

def rolling_autocorr_lag1(x, window, min_periods):
    lag = x.shift(1)
    result = {}
    for col in x.columns:
        result[col] = x[col].rolling(window=window, min_periods=min_periods).corr(lag[col])
    return pd.DataFrame(result, index=x.index)

def rolling_beta_with_intercept(x, y, window, min_periods, eps):
    rx = x.rolling(window, min_periods=min_periods)
    ry = y.rolling(window, min_periods=min_periods)
    ex = rx.mean()
    ey = ry.mean()
    exy = (x * y).rolling(window, min_periods=min_periods).mean()
    ex2 = (x * x).rolling(window, min_periods=min_periods).mean()
    cov_xy = exy - ex * ey
    var_x = ex2 - ex * ex
    beta = cov_xy / (var_x + eps)
    return beta.clip(lower=0.0)

def rolling_stress_threshold(stress, window, min_periods, q):
    return stress.shift(1).rolling(window, min_periods=min_periods).quantile(q)

def conditional_rolling_median_ratio(value, stress, threshold, window, min_periods, eps):
    is_stress = stress > threshold
    numerator = value.where(is_stress).rolling(window, min_periods=max(2, min_periods // 10)).median()
    denominator = value.rolling(window, min_periods=min_periods).median()
    return numerator / (denominator + eps)

def softplus_df(x):
    values = np.logaddexp(0.0, x.to_numpy(dtype=np.float64, copy=False))
    return pd.DataFrame(values, index=x.index, columns=x.columns)

def composite_positive_proxy(components, cfg):
    total = None
    count = None
    for component in components:
        z = robust_z_causal(component, cfg.primary_window, cfg.min_periods, cfg.eps)
        z = z.replace([np.inf, -np.inf], np.nan)
        valid = z.notna()
        if total is None:
            total = z.fillna(0.0)
            count = valid.astype('float64')
        else:
            total += z.fillna(0.0)
            count += valid.astype('float64')
    score = total / count.replace(0.0, np.nan)
    return softplus_df(score)

def recovery_speed_causal(gp0, stress, stress_threshold, baseline, tolerance, hold_bars, alpha, eps):
    g = gp0.to_numpy(dtype=np.float64, copy=False)
    s = stress.to_numpy(dtype=np.float64, copy=False)
    q = stress_threshold.to_numpy(dtype=np.float64, copy=False)
    b = baseline.to_numpy(dtype=np.float64, copy=False)
    T, N = g.shape
    output = np.full((T, N), np.nan)
    for j in range(N):
        in_event = False
        event_start = -1
        event_baseline = np.nan
        consecutive = 0
        duration_ewma = np.nan
        for t in range(T):
            if not in_event:
                if np.isfinite(s[t, j]) and np.isfinite(q[t, j]) and np.isfinite(b[t, j]) and (s[t, j] > q[t, j]) and (abs(b[t, j]) > eps):
                    in_event = True
                    event_start = t
                    event_baseline = b[t, j]
                    consecutive = 0
            elif np.isfinite(g[t, j]):
                distance = abs(g[t, j] - event_baseline) / (abs(event_baseline) + eps)
                if distance <= tolerance:
                    consecutive += 1
                else:
                    consecutive = 0
                if consecutive >= hold_bars:
                    recovery_start = t - hold_bars + 1
                    duration = max(1, recovery_start - event_start)
                    if np.isnan(duration_ewma):
                        duration_ewma = float(duration)
                    else:
                        duration_ewma = alpha * duration + (1 - alpha) * duration_ewma
                    in_event = False
                    event_start = -1
                    event_baseline = np.nan
                    consecutive = 0
            else:
                consecutive = 0
            if np.isfinite(duration_ewma):
                output[t, j] = 1.0 / (1.0 + duration_ewma)
    return pd.DataFrame(output, index=gp0.index, columns=gp0.columns)

cfg = FeatureConfig()

Define una sola configuración y los auxiliares originales. Medianas, cuantiles y escalas de referencia usan exclusivamente el pasado. Se mantienen los NaN de calentamiento; no se rellenan globalmente con cero. La supervivencia exige al menos dos eventos de estrés en la ventana diaria. Una ausencia interrumpe la confirmación consecutiva de recuperación.

In [ ]:
def validar_diario(frame):
    if not frame.columns.is_unique:
        raise ValueError('Columnas duplicadas.')
    out = frame.copy()
    if 'bar_end' not in out.columns:
        if out.index.name != 'bar_end':
            raise ValueError('Se requiere bar_end como columna o índice.')
        out = out.reset_index()
    out['bar_end'] = pd.to_datetime(out['bar_end'], utc=True, errors='raise')
    if out['bar_end'].isna().any() or out['bar_end'].duplicated().any():
        raise ValueError('Fechas ausentes o duplicadas.')
    out = out.sort_values('bar_end', kind='stable').reset_index(drop=True)
    if len(out) < 2 or not out['bar_end'].diff().iloc[1:].eq(pd.Timedelta(days=1)).all():
        raise ValueError('Se requiere una grilla diaria continua; no se imputan días faltantes.')
    return out


def reconstruir_reducido(original, ventana):
    cols = sorted(c for c in original.columns if c.endswith("_yp"))
    if not isinstance(ventana, int) or ventana < 1:
        raise ValueError("La ventana debe ser un entero positivo.")
    if len(cols) < 20 or len(original) <= ventana + 1:
        raise ValueError("Datos insuficientes para reconstruir el reducido.")

    p = original.loc[:, cols].to_numpy(dtype=np.float64, copy=True)
    totales = p.sum(axis=1, keepdims=True)
    if (
        not np.isfinite(p).all()
        or (p < 0).any()
        or not np.isfinite(totales).all()
        or (totales <= 0).any()
    ):
        raise ValueError("Participaciones inválidas; no se imputan automáticamente.")
    p /= totales

    scores = (
        pd.DataFrame(p)
        .rolling(ventana, min_periods=ventana)
        .mean()
        .shift(1)
        .iloc[ventana:]
        .to_numpy()
    )
    orden = np.argsort(-scores, axis=1, kind="stable")
    valores = np.take_along_axis(p[ventana:], orden, axis=1)
    n = len(cols)
    k5, k10, k25, k50 = [int(np.ceil(n * q)) for q in (.05, .10, .25, .50)]
    salida = {
        (f"RE__top05_pos_{j+1:03d}" if j < k5 else f"RE__top10_pos_{j+1:03d}"): valores[:, j]
        for j in range(k10)
    }
    grupos = (
        ("inferior_50", k50, n),
        ("intermedio_25", k25, k50),
        ("superior_15", k10, k25),
    )
    for nombre, inicio, fin in grupos:
        x = valores[:, inicio:fin]
        masa = x.sum(axis=1)
        q = np.divide(x, masa[:, None], out=np.zeros_like(x), where=masa[:, None] > 0)
        log_q = np.log(q, out=np.zeros_like(q), where=q > 0)
        salida[f"RE__{nombre}_masa"] = masa
        salida[f"RE__{nombre}_shannon_interno"] = -(q * log_q).sum(axis=1)
        salida[f"RE__{nombre}_hhi_interno"] = np.einsum("ij,ij->i", q, q)

    reducido = pd.DataFrame(salida)
    reducido.insert(0, "bar_end", original["bar_end"].iloc[ventana:].reset_index(drop=True))
    masa_total = valores[:, :k10].sum(axis=1)
    for nombre, _, _ in grupos:
        masa_total += reducido[f"RE__{nombre}_masa"].to_numpy()
    np.testing.assert_allclose(masa_total, 1.0, rtol=1e-12, atol=1e-12)
    return validar_diario(reducido)

df_original = validar_diario(pd.read_parquet(ORIGINAL))
df_reducido = reconstruir_reducido(df_original, VENTANA_RANKING)
columnas_yp = tuple(sorted(c for c in df_original if c.endswith('_yp')))
assets_originales = [c[:-3] for c in columnas_yp]

if len(columnas_yp) < 20:
    raise ValueError('Universo insuficiente para las particiones.')

requeridas = [base_col(a, k, cfg) for a in assets_originales
              for k in ('P', 'V', 'C', 'R', 'L', 'A', 'O', 'B', 'S', 'yp')]

if cfg.ys_col not in df_original:
    columnas_y = [f'{a}_y' for a in assets_originales]
    if not set(columnas_y).issubset(df_original.columns):
        raise ValueError('Falta YS y no están todas las columnas _y para obtener su suma.')
    df_original[cfg.ys_col] = df_original[columnas_y].sum(axis=1, min_count=len(columnas_y))

faltantes = sorted(set(requeridas) - set(df_original.columns))
if faltantes:
    raise ValueError(f'Faltan entradas originales: {faltantes[:20]}')

df = df_original.loc[:, ['bar_end', cfg.ys_col, *requeridas]].copy()
assets = assets_originales

if not all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns if c != 'bar_end'):
    raise TypeError('Las entradas de las fórmulas deben ser numéricas.')

if np.isinf(df.drop(columns='bar_end').to_numpy(dtype=np.float64)).any():
    raise ValueError('Hay infinitos en los datos originales.')

for sufijo in (cfg.observed_suffix, cfg.active_suffix):
    flags = df[[f'{a}{sufijo}' for a in assets]]
    if not flags.isin([0, 1]).all().all():
        raise ValueError(f'Indicadores inválidos o ausentes: {sufijo}')

print('Filas diarias:', len(df), '| Activos:', len(assets))

REDUCIDO = Path(ORIGINAL).with_name(Path(ORIGINAL).stem + "_red_con_fechas.parquet")
df_reducido.to_parquet(REDUCIDO, index=False)
print("Reducido regenerado:", df_reducido.shape)
print("Guardado con bar_end:", REDUCIDO)


Carga el original diario una sola vez y regenera el reducido desde sus participaciones _yp, con VENTANA_RANKING=3 y usando exclusivamente observaciones anteriores para ordenar. No lee el reducido sin fechas ni le asigna fechas por posición. Conserva Top 5%, Top 10% adicional y masa, Shannon interno y HHI de los tres grupos; bar_end se guarda como columna en <nombre_original>_red_con_fechas.parquet. El original y el reducido anterior se conservan.

La comprobación posterior de proyección utiliza este mismo df_reducido. Se mantienen las validaciones de fechas, continuidad, tipos e indicadores. Regenerar las fechas no elimina información futura que ya se haya introducido mediante interpolaciones en el original.

# Variables de Segundo Orden, 1ra generacion

## 1. Variables sistémicas

In [ ]:
n = len(df)
sys_active = np.zeros(n)
sys_trades = np.zeros(n)
sys_buy = np.zeros(n)
sys_sell = np.zeros(n)
for chunk in chunks(assets, cfg.asset_block_size):
    O = load_base(df, chunk, 'O', cfg)
    observed = O > 0
    A = load_base(df, chunk, 'A', cfg)
    C = load_base(df, chunk, 'C', cfg).where(observed)
    B = load_base(df, chunk, 'B', cfg).where(observed)
    S = load_base(df, chunk, 'S', cfg).where(observed)
    sys_active += A.sum(axis=1).to_numpy()
    sys_trades += C.sum(axis=1).to_numpy()
    sys_buy += B.sum(axis=1).to_numpy()
    sys_sell += S.sum(axis=1).to_numpy()
sys_flow = sys_buy + sys_sell
SYS_imbalance = (sys_buy - sys_sell) / (sys_flow + cfg.eps)
YS = df[cfg.ys_col].astype('float64')
YS_log = np.log(YS.clip(lower=cfg.eps))
YS_growth = YS_log.diff()
YS_volatility = YS_growth.rolling(cfg.primary_window, min_periods=cfg.min_periods).std()
system_features = pd.DataFrame({'SYS_active_nodes': sys_active, 'SYS_trades': sys_trades, 'SYS_imbalance': SYS_imbalance, 'YS_log': YS_log, 'YS_growth': YS_growth, 'YS_volatility': YS_volatility}, index=df.index).astype(cfg.dtype)
df = merge_blocks(df, [system_features])

Conserva las fórmulas originales sobre series de cada activo. Cada barra equivale a un día; las estadísticas usan ventanas retrospectivas y los resultados de los bloques se incorporan en un solo concat por etapa.

## 2. Precio y volumen físico

In [ ]:
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    O = load_base(df, chunk, 'O', cfg)
    A = load_base(df, chunk, 'A', cfg)
    observed = O > 0
    P = load_base(df, chunk, 'P', cfg).where(observed)
    V = load_base(df, chunk, 'V', cfg).where(observed)
    logP = np.log(P.where(P > 0))
    ret = logP.diff(cfg.return_lag)
    volatility = ret.rolling(cfg.primary_window, min_periods=cfg.min_periods).std()
    V_valid = V.where((V > 0) & (A > 0))
    dlogV = np.log(V_valid).diff()
    volume_dispersion = rolling_robust_dispersion(dlogV, cfg.primary_window, cfg.min_periods, cfg.eps)
    VSTAB = np.exp(-volume_dispersion)
    blocks.extend([name_block(ret, 'return_k1', cfg), name_block(volatility, 'rolling_volatility', cfg), name_block(dlogV, 'dlogV', cfg), name_block(VSTAB, 'VSTAB', cfg)])
df = merge_blocks(df, blocks)

Conserva las fórmulas originales sobre series de cada activo. Cada barra equivale a un día; las estadísticas usan ventanas retrospectivas y los resultados de los bloques se incorporan en un solo concat por etapa.

## 3. Actividad y continuidad

In [ ]:
blocks = []
SYS_trades = df['SYS_trades'].replace(0, np.nan)
for chunk in chunks(assets, cfg.asset_block_size):
    O = load_base(df, chunk, 'O', cfg)
    A = load_base(df, chunk, 'A', cfg)
    C = load_base(df, chunk, 'C', cfg)
    observed = O > 0
    C = C.where(observed)
    A_masked = A.where(observed)
    logC = np.log1p(C.clip(lower=0))
    TI = robust_z_causal(logC, cfg.primary_window, cfg.min_periods, cfg.eps)
    PERS_C = rolling_autocorr_lag1(logC, cfg.primary_window, cfg.min_periods)
    Arel = C.div(SYS_trades, axis=0)
    CA = A_masked.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean()
    CO = O.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean()
    blocks.extend([name_block(TI, 'TI', cfg), name_block(PERS_C, 'PERS_C', cfg), name_block(Arel, 'Arel', cfg), name_block(CA, 'CA', cfg), name_block(CO, 'CO', cfg)])
df = merge_blocks(df, blocks)

Conserva las fórmulas originales sobre series de cada activo. Cada barra equivale a un día; las estadísticas usan ventanas retrospectivas y los resultados de los bloques se incorporan en un solo concat por etapa.

## 4. Estructura buy/sell

In [ ]:
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    O = load_base(df, chunk, 'O', cfg)
    observed = O > 0
    B = load_base(df, chunk, 'B', cfg).where(observed)
    S = load_base(df, chunk, 'S', cfg).where(observed)
    logS1 = np.log1p(S.clip(lower=0))
    SI = robust_z_causal(logS1, cfg.primary_window, cfg.min_periods, cfg.eps)
    SG = np.log(S.clip(lower=0) + cfg.eps).diff()
    SP = SI.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean()
    total = B + S
    imbalance = (B - S) / (total + cfg.eps)
    sell_share = S / (total + cfg.eps)
    LR_BS = np.log((B + cfg.eps) / (S + cfg.eps))
    BIL = 2.0 * np.minimum(B, S) / (total + cfg.eps)
    blocks.extend([name_block(SI, 'SI', cfg), name_block(SG, 'SG', cfg), name_block(SP, 'SP', cfg), name_block(imbalance, 'imbalance', cfg), name_block(sell_share, 'sell_share', cfg), name_block(LR_BS, 'LR_BS', cfg), name_block(BIL, 'BIL', cfg)])
df = merge_blocks(df, blocks)

Conserva las fórmulas originales sobre series de cada activo. Cada barra equivale a un día; las estadísticas usan ventanas retrospectivas y los resultados de los bloques se incorporan en un solo concat por etapa.

## 5. Bilateralidad, absorción y liquidez

In [ ]:
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    O = load_base(df, chunk, 'O', cfg)
    observed = O > 0
    C = load_base(df, chunk, 'C', cfg).where(observed)
    S = load_base(df, chunk, 'S', cfg).where(observed)
    R = load_base(df, chunk, 'R', cfg).where(observed)
    L = load_base(df, chunk, 'L', cfg).where(observed)
    BIL = load_feature(df, chunk, 'BIL')
    BIL_disp = rolling_robust_dispersion(BIL.diff(), cfg.primary_window, cfg.min_periods, cfg.eps)
    BILST = BIL.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean() * np.exp(-BIL_disp)
    logC = np.log1p(C.clip(lower=0))
    logS = np.log1p(S.clip(lower=0))
    logL = np.log1p(L.clip(lower=0))
    ABS = logC / (R + cfg.eps)
    ABS_S = logS / (R + cfg.eps)
    RLIQ = logC / (1.0 + logL)
    RLIQ_S = logS / (1.0 + logL)
    LIQ = -logL
    blocks.extend([name_block(BILST, 'BILST', cfg), name_block(ABS, 'ABS', cfg), name_block(ABS_S, 'ABS_S', cfg), name_block(RLIQ, 'RLIQ', cfg), name_block(RLIQ_S, 'RLIQ_S', cfg), name_block(LIQ, 'LIQ', cfg)])
df = merge_blocks(df, blocks)

Conserva las fórmulas originales sobre series de cada activo. Cada barra equivale a un día; las estadísticas usan ventanas retrospectivas y los resultados de los bloques se incorporan en un solo concat por etapa.

## 6. Impacto, estrés y supervivencia

In [ ]:
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    O = load_base(df, chunk, 'O', cfg)
    observed = O > 0
    C = load_base(df, chunk, 'C', cfg).where(observed)
    R = load_base(df, chunk, 'R', cfg).where(observed)
    L = load_base(df, chunk, 'L', cfg).where(observed)
    B = load_base(df, chunk, 'B', cfg).where(observed)
    S = load_base(df, chunk, 'S', cfg).where(observed)
    ret = load_feature(df, chunk, 'return_k1')
    imbalance = load_feature(df, chunk, 'imbalance')
    logC = np.log1p(C.clip(lower=0))
    pressure = imbalance.abs() * logC
    beta = rolling_beta_with_intercept(pressure, ret.abs(), cfg.primary_window, cfg.min_periods, cfg.eps)
    IR = 1.0 / (beta + cfg.eps)
    zR = robust_z_causal(R, cfg.primary_window, cfg.min_periods, cfg.eps)
    zL = robust_z_causal(L, cfg.primary_window, cfg.min_periods, cfg.eps)
    zRet = robust_z_causal(ret.abs(), cfg.primary_window, cfg.min_periods, cfg.eps)
    STRESS = zR + zL + zRet
    threshold = rolling_stress_threshold(STRESS, cfg.primary_window, cfg.min_periods, cfg.stress_quantile)
    SSR = conditional_rolling_median_ratio(C, STRESS, threshold, cfg.primary_window, cfg.min_periods, cfg.eps)
    SSR_S = conditional_rolling_median_ratio(S, STRESS, threshold, cfg.primary_window, cfg.min_periods, cfg.eps)
    AST_S = S / (C + cfg.eps)
    blocks.extend([name_block(IR, 'IR', cfg), name_block(STRESS, 'STRESS', cfg), name_block(SSR, 'SSR', cfg), name_block(SSR_S, 'SSR_S', cfg), name_block(AST_S, 'AST_S', cfg)])
df = merge_blocks(df, blocks)

Conserva las fórmulas originales sobre series de cada activo. Cada barra equivale a un día; las estadísticas usan ventanas retrospectivas y los resultados de los bloques se incorporan en un solo concat por etapa.

## 7. gp inicial, recuperación y gp

In [ ]:
gp0_inputs = ('TI', 'CA', 'VSTAB', 'PERS_C', 'BIL', 'ABS', 'RLIQ', 'IR', 'SSR')
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    components = [load_feature(df, chunk, feature) for feature in gp0_inputs]
    gp0 = composite_positive_proxy(components, cfg)
    STRESS = load_feature(df, chunk, 'STRESS')
    threshold = rolling_stress_threshold(STRESS, cfg.primary_window, cfg.min_periods, cfg.stress_quantile)
    baseline = gp0.shift(1).rolling(cfg.primary_window, min_periods=cfg.min_periods).median()
    REC = recovery_speed_causal(gp0=gp0, stress=STRESS, stress_threshold=threshold, baseline=baseline, tolerance=cfg.recovery_tolerance, hold_bars=cfg.recovery_hold_bars, alpha=cfg.recovery_event_ewm_alpha, eps=cfg.eps)
    gp = composite_positive_proxy(components + [REC], cfg)
    chunk_blocks = [name_block(gp0, 'gp0', cfg), name_block(REC, 'REC', cfg), name_block(gp, 'gp', cfg)]
    blocks.extend(chunk_blocks)
df = merge_blocks(df, blocks)

Mantiene el cálculo por identidad y el historial continuo. No reinicia EWM ni eventos de recuperación en cortes arbitrarios; los bloques limitan columnas, no fechas.

## 8. gp multiescala

In [ ]:
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    gp = load_feature(df, chunk, 'gp')
    G = {}
    chunk_blocks = []
    for w in cfg.multiscale_windows:
        G[w] = gp.ewm(span=w, adjust=False, min_periods=max(2, min(w, cfg.min_periods))).mean()
        chunk_blocks.append(name_block(G[w], f'gp_W{w}', cfg))
    sum_log = None
    for w in cfg.multiscale_windows:
        lg = np.log(G[w].clip(lower=cfg.eps))
        sum_log = lg if sum_log is None else sum_log + lg
    fgp_nested = np.exp(sum_log / len(cfg.multiscale_windows))
    chunk_blocks.append(name_block(fgp_nested, 'fgp_nested', cfg))
    blocks.extend(chunk_blocks)
df = merge_blocks(df, blocks)

Mantiene el cálculo por identidad y el historial continuo. No reinicia EWM ni eventos de recuperación en cortes arbitrarios; los bloques limitan columnas, no fechas.

## 9. Estructura temporal de N

In [ ]:
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    gp = load_feature(df, chunk, 'gp')
    dlog_gp = np.log(gp.clip(lower=cfg.eps)).diff()
    gp_dispersion = rolling_robust_dispersion(dlog_gp, cfg.primary_window, cfg.min_periods, cfg.eps)
    NSTAB = np.exp(-gp_dispersion)
    NPERS = rolling_autocorr_lag1(gp, cfg.primary_window, cfg.min_periods)
    logs = []
    for w in cfg.multiscale_windows:
        G = load_feature(df, chunk, f'gp_W{w}')
        logs.append(np.log(G.clip(lower=cfg.eps)))
    stack = np.stack([x.to_numpy() for x in logs], axis=2)
    variance = np.nanvar(stack, axis=2)
    NCOH = pd.DataFrame(np.exp(-variance), index=df.index, columns=chunk)
    STRESS = load_feature(df, chunk, 'STRESS')
    threshold = rolling_stress_threshold(STRESS, cfg.primary_window, cfg.min_periods, cfg.stress_quantile)
    NRET = conditional_rolling_median_ratio(gp, STRESS, threshold, cfg.primary_window, cfg.min_periods, cfg.eps)
    chunk_blocks = [name_block(NSTAB, 'NSTAB', cfg), name_block(NPERS, 'NPERS', cfg), name_block(NCOH, 'NCOH', cfg), name_block(NRET, 'NRET', cfg)]
    blocks.extend(chunk_blocks)
df = merge_blocks(df, blocks)

Mantiene el cálculo por identidad y el historial continuo. No reinicia EWM ni eventos de recuperación en cortes arbitrarios; los bloques limitan columnas, no fechas.

## 10. Estructura competitiva y anidada de N

In [ ]:
A_all = load_base(df, assets, 'A', cfg)
gp_active = load_feature(df, assets, 'gp').where(A_all > 0)
NRANK = np.exp(-gp_active.rank(axis=1, pct=True, method='average').diff().abs())
blocks = [name_block(NRANK, 'NRANK', cfg)]
q_logs = []
for w in cfg.multiscale_windows:
    G = load_feature(df, assets, f'gp_W{w}').where(A_all > 0)
    total = G.sum(axis=1, min_count=1).replace(0, np.nan)
    qg = G.div(total, axis=0)
    blocks.append(name_block(qg, f'qg_W{w}', cfg))
    q_logs.append(np.log(qg.clip(lower=cfg.eps)).to_numpy())
stack = np.stack(q_logs, axis=2)
NEST = pd.DataFrame(np.exp(-np.nanvar(stack, axis=2)), index=df.index, columns=assets)
blocks.append(name_block(NEST, 'NEST', cfg))
df = merge_blocks(df, blocks)

Conserva el ranking competitivo y los simplex de resistencia sobre todo el universo activo. Los denominadores se calculan una vez por escala, no por cada bloque. No usa clasificaciones globales de todo el período.

## 11. N, energía

In [ ]:
N_inputs = ('NSTAB', 'NPERS', 'NCOH', 'NRET', 'NRANK', 'NEST')
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    components = [load_feature(df, chunk, feature) for feature in N_inputs]
    N_context = composite_positive_proxy(components, cfg)
    fgp = load_feature(df, chunk, 'fgp_nested')
    yp = load_base(df, chunk, 'yp', cfg)
    yp_safe = yp.clip(lower=cfg.eps, upper=1.0)
    surprise = -np.log(yp_safe)
    N_theory = fgp / (surprise + cfg.eps)
    gp = load_feature(df, chunk, 'gp')
    E_context = gp / (N_context + cfg.eps)
    E_theory = gp / (N_theory + cfg.eps)
    chunk_blocks = [name_block(N_context, 'N_context', cfg), name_block(N_theory, 'N_theory', cfg), name_block(E_context, 'E_context', cfg), name_block(E_theory, 'E_theory', cfg)]
    blocks.extend(chunk_blocks)
df = merge_blocks(df, blocks)

Conserva las fórmulas originales. Las diferencias y ventanas se calculan antes de trasladar los activos a puestos. Los objetivos futuros se construyen más adelante, en una tabla separada.

## 12. Derivadas de E_context

In [ ]:
E_all = load_feature(df, assets, 'E_context').astype('float32')
A_all = load_base(df, assets, 'A', cfg).astype('float32')
E_active = E_all.where(A_all > 0)
E_median = E_active.median(axis=1)
E_rank = E_active.rank(axis=1, pct=True, method='average')
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    E = E_all.loc[:, chunk]
    logE = np.log(E.clip(lower=cfg.eps))
    Erel = E.sub(E_median, axis=0)
    dE = E.diff()
    ddE = dE.diff()
    ESH = robust_z_causal(E, cfg.primary_window, cfg.min_periods, cfg.eps)
    ERANK = E_rank.loc[:, chunk]
    chunk_blocks = [name_block(logE, 'Ectx_logE', cfg), name_block(Erel, 'Ectx_Erel', cfg), name_block(dE, 'Ectx_dE', cfg), name_block(ddE, 'Ectx_ddE', cfg), name_block(ESH, 'Ectx_ESH', cfg), name_block(ERANK, 'Ectx_ERANK', cfg)]
    blocks.extend(chunk_blocks)
df = merge_blocks(df, blocks)

Conserva las fórmulas originales. Las diferencias y ventanas se calculan antes de trasladar los activos a puestos. Los objetivos futuros se construyen más adelante, en una tabla separada.

## 13. Derivadas de E_theory

In [ ]:
E_all = load_feature(df, assets, 'E_theory').astype('float32')
A_all = load_base(df, assets, 'A', cfg).astype('float32')
E_active = E_all.where(A_all > 0)
E_median = E_active.median(axis=1)
E_rank = E_active.rank(axis=1, pct=True, method='average')
blocks = []
for chunk in chunks(assets, cfg.asset_block_size):
    E = E_all.loc[:, chunk]
    logE = np.log(E.clip(lower=cfg.eps))
    Erel = E.sub(E_median, axis=0)
    dE = E.diff()
    ddE = dE.diff()
    ESH = robust_z_causal(E, cfg.primary_window, cfg.min_periods, cfg.eps)
    ERANK = E_rank.loc[:, chunk]
    chunk_blocks = [name_block(logE, 'Eth_logE', cfg), name_block(Erel, 'Eth_Erel', cfg), name_block(dE, 'Eth_dE', cfg), name_block(ddE, 'Eth_ddE', cfg), name_block(ESH, 'Eth_ESH', cfg), name_block(ERANK, 'Eth_ERANK', cfg)]
    blocks.extend(chunk_blocks)
df = merge_blocks(df, blocks)

Conserva las fórmulas originales. Las diferencias y ventanas se calculan antes de trasladar los activos a puestos. Los objetivos futuros se construyen más adelante, en una tabla separada.

## Proyección causal al dataframe reducido

In [ ]:
fechas_originales = pd.DatetimeIndex(df['bar_end'])
fechas_red = pd.DatetimeIndex(df_reducido['bar_end'])
filas_red = fechas_originales.get_indexer(fechas_red)
if (filas_red < VENTANA_RANKING).any():
    raise ValueError('Faltan fechas originales o historia para reproducir el ranking.')
P = df_original.loc[:, list(columnas_yp)].to_numpy(dtype=np.float64)
if not np.isfinite(P).all() or (P < 0).any() or (P.sum(axis=1) <= 0).any():
    raise ValueError('Participaciones inválidas; no se imputan ni inventan identidades.')
P /= P.sum(axis=1, keepdims=True)
scores = pd.DataFrame(P).rolling(VENTANA_RANKING, min_periods=VENTANA_RANKING).mean().shift(1)
orden = np.argsort(-scores.to_numpy()[filas_red], axis=1, kind='stable')
n = len(columnas_yp)
k5, k10, k25, k50 = [int(np.ceil(n * q)) for q in (.05, .10, .25, .50)]
puestos = [f'RE__top05_pos_{j+1:03d}' if j < k5 else f'RE__top10_pos_{j+1:03d}' for j in range(k10)]
grupos = [('inferior_50', k50, n), ('intermedio_25', k25, k50), ('superior_15', k10, k25)]
esperadas = puestos + [f'RE__{g}_{m}' for g, _, _ in grupos for m in ('masa','shannon_interno','hhi_interno')]
if set(df_reducido.columns) != set(['bar_end', *esperadas]):
    raise ValueError('El esquema reducido no coincide con los puestos y contextos esperados.')
ordenadas = np.take_along_axis(P[filas_red], orden, axis=1)
np.testing.assert_allclose(df_reducido[puestos], ordenadas[:, :k10], rtol=1e-6, atol=1e-10)
for grupo, inicio, fin in grupos:
    x = ordenadas[:, inicio:fin]
    masa = x.sum(axis=1)
    q = np.divide(x, masa[:,None], out=np.zeros_like(x), where=masa[:,None] > 0)
    logq = np.log(q, out=np.zeros_like(q), where=q > 0)
    esperado = np.column_stack([masa, -(q*logq).sum(axis=1), (q*q).sum(axis=1)])
    cols = [f'RE__{grupo}_{m}' for m in ('masa','shannon_interno','hhi_interno')]
    np.testing.assert_allclose(df_reducido[cols], esperado, rtol=1e-6, atol=1e-10)

seleccion = orden[:, :k10]
identidades_red = pd.DataFrame(np.asarray(assets_originales)[seleccion], index=fechas_red, columns=puestos)
features_por_activo = sorted(c[len(assets_originales[0]) + 2:] for c in df.columns if c.startswith(assets_originales[0] + '__'))
bloques_red = [df_reducido.set_index('bar_end')]
for feature in features_por_activo:
    valores = load_feature(df, assets_originales, feature).to_numpy()[filas_red]
    elegidos = np.take_along_axis(valores, seleccion, axis=1)
    bloques_red.append(pd.DataFrame(elegidos, index=fechas_red, columns=[f'{p}__{feature}' for p in puestos]))
for tipo in ('P','V','C','R','L','A','O','B','S'):
    valores = load_base(df, assets_originales, tipo, cfg).to_numpy()[filas_red]
    elegidos = np.take_along_axis(valores, seleccion, axis=1)
    bloques_red.append(pd.DataFrame(elegidos, index=fechas_red, columns=[base_col(p, tipo, cfg) for p in puestos]))
sistemicas = [c for c in df if c.startswith('SYS_') or c.startswith('YS_') or c == cfg.ys_col]
bloques_red.append(df.set_index('bar_end').loc[fechas_red, sistemicas])
df_red = pd.concat(bloques_red, axis=1)
df_red.index.name = 'bar_end'
if not df_red.columns.is_unique or df_red.columns.str.contains('TARGET').any():
    raise ValueError('Columnas duplicadas u objetivos mezclados con entradas.')
print('Características finales:', df_red.shape)

Reconstruye exactamente el ranking de tres observaciones anteriores y verifica también los tres grupos contra el parquet reducido. Si el esquema, las fechas o la ventana difieren, detiene el proceso. Conserva las 69 columnas RE__, incorpora las variables originales seleccionadas y añade sus derivadas. La identidad se guarda aparte. Las estadísticas comprimidas se mantienen como contexto, nunca se hacen pasar por precios, volumen o activos independientes.

In [ ]:
objetivos = {}
for h in HORIZONTES:
    futuras = filas_red + h
    validas = futuras < len(P)
    valores = np.full(seleccion.shape, np.nan)
    valores[validas] = np.take_along_axis(P[futuras[validas]], seleccion[validas], axis=1)
    for j, puesto in enumerate(puestos):
        objetivos[f'{puesto}__TARGET_yp_t{h}'] = valores[:, j]
df_targets = pd.DataFrame(objetivos, index=df_red.index)

def indices_cronologicos(indice, horizonte=1, train_frac=.70, val_frac=.15):
    if horizonte < 1 or train_frac <= 0 or val_frac <= 0 or train_frac + val_frac >= 1:
        raise ValueError('Parámetros de split inválidos.')
    n = len(indice)
    a, b = int(n * train_frac), int(n * (train_frac + val_frac))
    cortes = {'train': (0, a), 'val': (a, b), 'test': (b, n)}
    salida = {nombre: np.arange(inicio, max(inicio, fin-horizonte)) for nombre, (inicio, fin) in cortes.items()}
    if any(len(x) == 0 for x in salida.values()):
        raise ValueError('Una partición quedó vacía.')
    return salida

def walk_forward(indice, horizonte=1, train_dias=365, val_dias=60, paso_dias=60, limite=None):
    fin = len(indice) if limite is None else min(limite, len(indice))
    if min(horizonte, train_dias, val_dias, paso_dias) < 1 or min(train_dias, val_dias) <= horizonte:
        raise ValueError('Ventanas walk-forward inválidas.')
    for corte in range(train_dias, fin-val_dias+1, paso_dias):
        yield {'train': np.arange(0, corte-horizonte),
               'val': np.arange(corte, corte+val_dias-horizonte)}

SPLITS = indices_cronologicos(df_red.index, horizonte=max(HORIZONTES))
inicio_test = int(len(df_red) * .85)
WALK_FORWARD = list(walk_forward(df_red.index, horizonte=max(HORIZONTES), limite=inicio_test))
display(pd.DataFrame({k: {'filas':len(v), 'desde':df_red.index[v[0]], 'hasta':df_red.index[v[-1]]} for k,v in SPLITS.items()}).T)

El objetivo de cada puesto sigue al activo que ocupaba ese puesto en la fecha de origen; no al ocupante futuro. No se renormaliza sobre el Top 10%. Los splits purgan las últimas h muestras de cada partición para que ninguna etiqueta cruce el corte. El walk-forward expansivo queda antes del test final. Las filas de calentamiento mantienen NaN: su imputación, si se decide usarla, se ajusta solo dentro del entrenamiento de cada fold.

In [ ]:
df_red.to_parquet(SALIDA, index=True)
df_targets.to_parquet(RUTA / '06-2026 - 07-2023_red_targets_diarios.parquet', index=True)
identidades_red.to_parquet(RUTA / '06-2026 - 07-2023_red_identidades.parquet', index=True)
manifiesto = {'configuracion': asdict(cfg), 'ventana_ranking': VENTANA_RANKING,
              'columnas_yp': list(columnas_yp), 'horizontes_dias': list(HORIZONTES),
              'caracteristicas': list(df_red.columns), 'universo_original': len(assets_originales),
              'disponibilidad': 'al cierre de bar_end', 'target': 'mismo activo de la fecha de origen'}
(RUTA / '06-2026 - 07-2023_red_features_manifest.json').write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')
print(SALIDA)

Guarda características, objetivos e identidades en archivos separados y conserva el índice temporal. El manifiesto permite reproducir esquema, ventanas y disponibilidad. En producción no se necesitan los objetivos; se conserva el historial de las series por activo y el estado de EWM/recuperación, o se recalcula el prefijo completo. Tres días de historial alcanzan para el ranking, pero no para toda la ingeniería.

## Experimentos y evaluación cronológica
Se conservan las auditorías, redundancia, circularidad, estabilidad, estacionariedad, bootstrap y modelos por familias. La exploración usa entrenamiento y la selección usa validación; el test queda reservado para una única evaluación del modelo elegido. Las métricas simplex de este bloque describen la distribución condicional entre los activos seleccionados en la fecha de origen, no el universo completo.

In [ ]:
EJECUTAR_EXPERIMENTOS = False
cfg_modelo = replace(cfg, yp_suffix='')
df_modelo = df_red.reset_index()
assets_modelo = puestos

Las celdas de ingeniería anteriores forman el dataset completo. Los experimentos opcionales se activan con este interruptor, sin reabrir archivos ni reiniciar el historial.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import wasserstein_distance
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
NODE_FEATURES = {'return_k1': 'price_volume', 'rolling_volatility': 'price_volume', 'dlogV': 'price_volume', 'VSTAB': 'price_volume', 'TI': 'activity', 'PERS_C': 'activity', 'Arel': 'activity', 'CA': 'activity', 'CO': 'activity', 'SI': 'flow', 'SG': 'flow', 'SP': 'flow', 'imbalance': 'flow', 'sell_share': 'flow', 'LR_BS': 'flow', 'BIL': 'flow', 'BILST': 'absorption_liquidity', 'ABS': 'absorption_liquidity', 'ABS_S': 'absorption_liquidity', 'RLIQ': 'absorption_liquidity', 'RLIQ_S': 'absorption_liquidity', 'LIQ': 'absorption_liquidity', 'IR': 'stress_resistance', 'STRESS': 'stress_resistance', 'SSR': 'stress_resistance', 'SSR_S': 'stress_resistance', 'AST_S': 'stress_resistance', 'FLOW_TOTAL_AUDIT': 'audit_only', 'TURNOVER_PER_TRADE_AUDIT': 'audit_only'}
SYSTEM_FEATURES = ['SYS_active_nodes', 'SYS_trades', 'SYS_imbalance', 'YS_log', 'YS_growth', 'YS_volatility']
MODEL_FAMILIES = {'price_volume': ['return_k1', 'rolling_volatility', 'dlogV', 'VSTAB'], 'activity': ['TI', 'PERS_C', 'Arel', 'CA', 'CO'], 'flow': ['SI', 'SG', 'SP', 'imbalance', 'sell_share', 'LR_BS', 'BIL'], 'absorption_liquidity': ['BILST', 'ABS', 'ABS_S', 'RLIQ', 'RLIQ_S', 'LIQ'], 'stress_resistance': ['IR', 'STRESS', 'SSR', 'SSR_S', 'AST_S']}
SUPPORT = {'return_k1': (None, None), 'rolling_volatility': (0.0, None), 'dlogV': (None, None), 'VSTAB': (0.0, 1.0), 'TI': (None, None), 'PERS_C': (-1.0, 1.0), 'Arel': (0.0, 1.0), 'CA': (0.0, 1.0), 'CO': (0.0, 1.0), 'SI': (None, None), 'SG': (None, None), 'SP': (None, None), 'imbalance': (-1.0, 1.0), 'sell_share': (0.0, 1.0), 'LR_BS': (None, None), 'BIL': (0.0, 1.0), 'BILST': (0.0, 1.0), 'ABS': (0.0, None), 'ABS_S': (0.0, None), 'RLIQ': (0.0, None), 'RLIQ_S': (0.0, None), 'LIQ': (None, 0.0), 'IR': (0.0, None), 'STRESS': (None, None), 'SSR': (0.0, None), 'SSR_S': (0.0, None), 'AST_S': (0.0, None), 'FLOW_TOTAL_AUDIT': (0.0, None), 'TURNOVER_PER_TRADE_AUDIT': (0.0, None)}
STRICT_FEATURES = [f for f, family in NODE_FEATURES.items() if family != 'audit_only']
NODE_FEATURES = {f: family for f, family in NODE_FEATURES.items() if family != 'audit_only'}
SYSTEM_FEATURES += [c for c in df_red if c.startswith('RE__') and '_pos_' not in c]

Mantiene las cinco familias experimentales y añade las nueve estadísticas comprimidas como contexto sistémico. Excluye del catálogo las dos variables de auditoría que el código original no generaba.

In [ ]:
def audit_node_feature(df, feature, assets, cfg, sample_rows=5000, seed=42):
    rng = np.random.default_rng(seed)
    if len(df) > sample_rows:
        row_sample = np.sort(rng.choice(len(df), sample_rows, replace=False))
    else:
        row_sample = np.arange(len(df))
    total = 0
    nan_count = 0
    inf_count = 0
    zero_count = 0
    support_bad = 0
    constant_assets = 0
    asset_count = 0
    sample_values = []
    lower, upper = SUPPORT.get(feature, (None, None))
    for chunk in chunks(assets, cfg.asset_block_size):
        X = load_feature(df, chunk, feature)
        arr = X.to_numpy(dtype=np.float64, copy=False)
        total += arr.size
        nan_count += np.isnan(arr).sum()
        inf_count += np.isinf(arr).sum()
        finite = np.isfinite(arr)
        zero_count += ((arr == 0) & finite).sum()
        if lower is not None:
            support_bad += ((arr < lower - 1e-07) & finite).sum()
        if upper is not None:
            support_bad += ((arr > upper + 1e-07) & finite).sum()
        for j in range(arr.shape[1]):
            x = arr[:, j]
            x = x[np.isfinite(x)]
            asset_count += 1
            if len(x) == 0 or np.nanstd(x) < 1e-12:
                constant_assets += 1
        sampled = arr[row_sample, :].ravel()
        sampled = sampled[np.isfinite(sampled)]
        if len(sampled):
            sample_values.append(sampled)
    if sample_values:
        sample_values = np.concatenate(sample_values)
        q = np.quantile(sample_values, [0.001, 0.01, 0.5, 0.99, 0.999])
        q001, q01, med, q99, q999 = q
    else:
        q001 = q01 = med = q99 = q999 = np.nan
    return {'feature': feature, 'family': NODE_FEATURES[feature], 'nan_pct': 100 * nan_count / total, 'inf_pct': 100 * inf_count / total, 'zero_pct': 100 * zero_count / total, 'support_violation_pct': 100 * support_bad / total, 'constant_asset_pct': 100 * constant_assets / max(asset_count, 1), 'q_0.001': q001, 'q_0.01': q01, 'median': med, 'q_0.99': q99, 'q_0.999': q999}

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def audit_system_features(df, features):
    rows = []
    for feature in features:
        x = df[feature].to_numpy(dtype=np.float64, copy=False)
        finite = np.isfinite(x)
        xf = x[finite]
        rows.append({'feature': feature, 'nan_pct': 100 * np.isnan(x).mean(), 'inf_pct': 100 * np.isinf(x).mean(), 'zero_pct': 100 * ((x == 0) & finite).mean(), 'median': np.nanmedian(xf) if len(xf) else np.nan, 'IQR': np.nanquantile(xf, 0.75) - np.nanquantile(xf, 0.25) if len(xf) else np.nan, 'std': np.nanstd(xf) if len(xf) else np.nan})
    return pd.DataFrame(rows)

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def sampled_node_panel(df, assets, features, cfg, n_times=3000, n_assets=96, seed=42):
    rng = np.random.default_rng(seed)
    chosen_assets = list(assets) if len(assets) <= n_assets else list(rng.choice(assets, n_assets, replace=False))
    valid_rows = np.arange(0, len(df) - 1)
    if len(valid_rows) > n_times:
        time_idx = np.sort(rng.choice(valid_rows, n_times, replace=False))
    else:
        time_idx = valid_rows
    data = {}
    for feature in features:
        X = load_feature(df, chosen_assets, feature)
        data[feature] = X.iloc[time_idx].to_numpy(dtype=np.float32, copy=False).reshape(-1)
    yp_cols = [f'{a}{cfg.yp_suffix}' for a in chosen_assets]
    yp = df.loc[df.index[time_idx], yp_cols].to_numpy(dtype=np.float32, copy=False).reshape(-1)
    panel = pd.DataFrame(data)
    panel['yp_t'] = yp
    panel['log_yp_t'] = np.log(np.clip(yp, cfg.eps, None))
    return panel

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def redundancy_report(panel, features, threshold=0.97):
    X = panel[features]
    corr = X.corr(method='spearman')
    rows = []
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            a = features[i]
            b = features[j]
            rho = corr.loc[a, b]
            if np.isfinite(rho) and abs(rho) >= threshold:
                rows.append({'feature_1': a, 'feature_2': b, 'spearman': rho})
    return pd.DataFrame(rows).sort_values('spearman', key=np.abs, ascending=False) if rows else pd.DataFrame(columns=['feature_1', 'feature_2', 'spearman'])

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def circularity_report(panel, features, max_samples=50000, seed=42):
    rng = np.random.default_rng(seed)
    work = panel[features + ['log_yp_t']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(work) > max_samples:
        idx = rng.choice(len(work), max_samples, replace=False)
        work = work.iloc[np.sort(idx)]
    if len(work) < 10:
        return pd.DataFrame(columns=['feature', 'family', 'spearman_logyp', 'MI_logyp', 'linear_R2_logyp'])
    y = work['log_yp_t'].to_numpy()
    rows = []
    for feature in features:
        x = work[feature].to_numpy()
        rho = pd.Series(x).corr(pd.Series(y), method='spearman')
        mi = mutual_info_regression(x.reshape(-1, 1), y, random_state=seed)[0]
        split = int(len(x) * 0.7)
        model = LinearRegression()
        model.fit(x[:split].reshape(-1, 1), y[:split])
        pred = model.predict(x[split:].reshape(-1, 1))
        r2 = r2_score(y[split:], pred)
        rows.append({'feature': feature, 'family': NODE_FEATURES[feature], 'spearman_logyp': rho, 'MI_logyp': mi, 'linear_R2_logyp': r2})
    return pd.DataFrame(rows).sort_values('MI_logyp', ascending=False).reset_index(drop=True)

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def temporal_stability(df, assets, features, cfg, n_blocks=4, n_assets=64, rows_per_block=3000, seed=42):
    rng = np.random.default_rng(seed)
    chosen_assets = list(assets) if len(assets) <= n_assets else list(rng.choice(assets, n_assets, replace=False))
    blocks_idx = np.array_split(np.arange(len(df)), n_blocks)
    output = []
    for feature in features:
        distributions = []
        for b, idx in enumerate(blocks_idx):
            if len(idx) > rows_per_block:
                sample_idx = np.sort(rng.choice(idx, rows_per_block, replace=False))
            else:
                sample_idx = idx
            X = load_feature(df, chosen_assets, feature)
            values = X.iloc[sample_idx].to_numpy(dtype=np.float32, copy=False).reshape(-1)
            values = values[np.isfinite(values)]
            distributions.append(values)
        if any(len(v) == 0 for v in distributions):
            continue
        base = distributions[0]
        base_iqr = np.quantile(base, 0.75) - np.quantile(base, 0.25)
        for b, values in enumerate(distributions):
            med = np.median(values)
            iqr = np.quantile(values, 0.75) - np.quantile(values, 0.25)
            if b == 0:
                w_norm = 0.0
            else:
                w = wasserstein_distance(base, values)
                w_norm = w / (base_iqr + cfg.eps)
            output.append({'feature': feature, 'block': b, 'median': med, 'IQR': iqr, 'wasserstein_vs_block0_IQR': w_norm})
    return pd.DataFrame(output)

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def stationarity_screen(df, assets, features, cfg):
    rows = []
    for feature in features:
        X = load_feature(df, assets, feature)
        x = X.median(axis=1, skipna=True).replace([np.inf, -np.inf], np.nan).dropna()
        if len(x) < 100:
            continue
        try:
            adf_p = adfuller(x, autolag='AIC')[1]
        except Exception:
            adf_p = np.nan
        try:
            kpss_p = kpss(x, regression='c', nlags='auto')[1]
        except Exception:
            kpss_p = np.nan
        rows.append({'feature': feature, 'ADF_p': adf_p, 'KPSS_p': kpss_p})
    return pd.DataFrame(rows)

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def normalize_simplex(X, eps=1e-12, smooth=False):
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X = np.clip(X, 0.0, None)
    if smooth:
        X = X + eps
    denom = X.sum(axis=1, keepdims=True)
    return X / np.maximum(denom, eps)

def simplex_metrics(y, q, eps=1e-12):
    y = normalize_simplex(y, eps, smooth=False)
    q = normalize_simplex(q, eps, smooth=True)
    log_y = np.log(np.clip(y, eps, None))
    log_q = np.log(np.clip(q, eps, None))
    CE_t = -np.sum(y * log_q, axis=1)
    KL_t = np.sum(y * (log_y - log_q), axis=1)
    m = 0.5 * (y + q)
    log_m = np.log(np.clip(m, eps, None))
    JS_t = 0.5 * (np.sum(y * (log_y - log_m), axis=1) + np.sum(q * (log_q - log_m), axis=1))
    return {'CE': CE_t.mean(), 'KL': KL_t.mean(), 'JS': JS_t.mean(), 'MAE': np.mean(np.abs(y - q)), 'RMSE': np.sqrt(np.mean((y - q) ** 2)), 'CE_t': CE_t}

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def sample_rows_evenly(idx, max_rows):
    idx = np.asarray(idx)
    if len(idx) <= max_rows:
        return idx
    positions = np.linspace(0, len(idx) - 1, max_rows, dtype=int)
    return idx[positions]

def load_feature_matrix(df, assets, feature, rows):
    cols = [f'{a}__{feature}' for a in assets]
    return df.iloc[rows][cols].to_numpy(dtype=np.float64, copy=False)

def fit_robust_scaler(X, eps=1e-12):
    values = X[np.isfinite(X)]
    if not len(values):
        raise ValueError('La variable no tiene valores finitos en entrenamiento.')
    median = np.median(values)
    q25, q75 = np.quantile(values, [0.25, 0.75])
    scale = (q75 - q25) / 1.349
    return (median, max(scale, eps))

def transform_feature(X, median, scale):
    Z = (X - median) / scale
    Z = np.nan_to_num(Z, nan=0.0, posinf=8.0, neginf=-8.0)
    return np.clip(Z, -8.0, 8.0)

def tilted_simplex(p, x, beta, eps=1e-12):
    p = normalize_simplex(p, eps, smooth=True)
    logits = np.log(np.clip(p, eps, None)) + beta * x
    return softmax(logits, axis=1)

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def moving_block_bootstrap(x, block_size=7, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < block_size:
        raise ValueError('Serie demasiado corta.')
    means = np.empty(n_boot)
    max_start = n - block_size
    n_blocks = int(np.ceil(n / block_size))
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks)
        sample = np.concatenate([x[s:s + block_size] for s in starts])[:n]
        means[b] = sample.mean()
    return {'mean': x.mean(), 'CI_2.5': np.quantile(means, 0.025), 'CI_97.5': np.quantile(means, 0.975), 'P_mean_gt_0': np.mean(means > 0)}

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def build_family_panel(df, assets, rows, node_features, system_features=None, cfg=None, horizon=1):
    rows = np.asarray(rows)
    rows = rows[rows + horizon < len(df)]
    N = len(assets)
    T = len(rows)
    X_parts = {}
    for feature in node_features:
        M = load_feature_matrix(df, assets, feature, rows)
        X_parts[feature] = M.reshape(-1)
    p = yp_matrix(df, assets, rows, cfg, horizon=0)
    y = yp_matrix(df, assets, rows, cfg, horizon=horizon)
    p = normalize_simplex(p, cfg.eps, smooth=True)
    y = normalize_simplex(y, cfg.eps, smooth=True)
    log_p = np.log(np.clip(p, cfg.eps, None))
    delta_target = np.log(np.clip(y, cfg.eps, None)) - log_p
    X_parts['log_yp_t'] = log_p.reshape(-1)
    if system_features:
        for feature in system_features:
            values = df[feature].iloc[rows].to_numpy(dtype=np.float32, copy=False)
            X_parts[f'SYS__{feature}'] = np.repeat(values, N)
    X = pd.DataFrame(X_parts)
    target = delta_target.reshape(-1)
    metadata = {'rows': rows, 'n_times': T, 'n_assets': N, 'p': p, 'y': y}
    valid = np.isfinite(target)
    return (X.loc[valid].reset_index(drop=True), target[valid], metadata, valid)

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def evaluate_family_delta(delta_prediction, metadata, valid, eps=1e-12):
    T = metadata['n_times']
    N = metadata['n_assets']
    full_delta = np.zeros(T * N, dtype=np.float64)
    full_delta[valid] = delta_prediction
    delta = full_delta.reshape(T, N)
    p = metadata['p']
    y = metadata['y']
    logits = np.log(np.clip(p, eps, None)) + delta
    q = softmax(logits, axis=1)
    baseline = simplex_metrics(y, p, eps)
    model = simplex_metrics(y, q, eps)
    return {'CE_baseline': baseline['CE'], 'CE_model': model['CE'], 'delta_CE': baseline['CE'] - model['CE'], 'KL': model['KL'], 'JS': model['JS'], 'MAE': model['MAE'], 'RMSE': model['RMSE']}

Define auxiliares experimentales sin ejecutarlos. Las llamadas de exploración posteriores se limitan a entrenamiento; el bootstrap usa bloques de siete días como parámetro inicial.

In [ ]:
def yp_matrix(df, assets, rows, cfg, horizon=0):
    rows = np.asarray(rows, dtype=int)
    if horizon < 0 or horizon not in (0, *HORIZONTES):
        raise ValueError('Horizonte no configurado.')
    fechas = pd.DatetimeIndex(df[cfg.time_col].iloc[rows])
    posiciones_red = fechas_red.get_indexer(fechas)
    if (posiciones_red < 0).any():
        raise ValueError('Fechas ajenas al reducido.')
    posiciones_puesto = pd.Index(puestos).get_indexer(assets)
    if (posiciones_puesto < 0).any():
        raise ValueError('Puestos desconocidos.')
    origen = filas_red[posiciones_red]
    identidades = seleccion[posiciones_red][:, posiciones_puesto]
    if (origen + horizon >= len(P)).any():
        raise ValueError('Objetivo fuera del historial disponible.')
    return np.take_along_axis(P[origen + horizon], identidades, axis=1)

Accede al objetivo del mismo activo de origen para cualquier horizonte configurado. No usa shift sobre columnas de puestos que cambian de identidad.

In [ ]:
def screen_single_feature(df, assets, feature, cfg, splits, horizon=1, train_max_rows=15000, evaluation='val'):
    train_rows = sample_rows_evenly(splits['train'], train_max_rows)
    train_rows = train_rows[train_rows + horizon < len(df)]
    X_train = load_feature_matrix(df, assets, feature, train_rows)
    med, scale = fit_robust_scaler(X_train, cfg.eps)
    X_train = transform_feature(X_train, med, scale)
    p_train = yp_matrix(df, assets, train_rows, cfg, horizon=0)
    y_train = yp_matrix(df, assets, train_rows, cfg, horizon=horizon)
    p_train = normalize_simplex(p_train, cfg.eps, smooth=True)
    y_train = normalize_simplex(y_train, cfg.eps, smooth=False)

    def objective(beta):
        q = tilted_simplex(p_train, X_train, beta, cfg.eps)
        return simplex_metrics(y_train, q, cfg.eps)['CE']
    result = minimize_scalar(objective, bounds=(-5.0, 5.0), method='bounded')
    beta = result.x
    if evaluation not in ('val', 'test'):
        raise ValueError('Evaluación inválida.')
    test_rows = splits[evaluation]
    test_rows = test_rows[test_rows + horizon < len(df)]
    X_test = load_feature_matrix(df, assets, feature, test_rows)
    X_test = transform_feature(X_test, med, scale)
    p_test = yp_matrix(df, assets, test_rows, cfg, horizon=0)
    y_test = yp_matrix(df, assets, test_rows, cfg, horizon=horizon)
    p_test = normalize_simplex(p_test, cfg.eps, smooth=True)
    y_test = normalize_simplex(y_test, cfg.eps, smooth=False)
    baseline_metrics = simplex_metrics(y_test, p_test, cfg.eps)
    q_test = tilted_simplex(p_test, X_test, beta, cfg.eps)
    feature_metrics = simplex_metrics(y_test, q_test, cfg.eps)
    return {'feature': feature, 'family': NODE_FEATURES[feature], 'horizon_bars': horizon, 'horizon_days': horizon, 'evaluation': evaluation, 'beta': beta, 'CE_baseline': baseline_metrics['CE'], 'CE_feature': feature_metrics['CE'], 'delta_CE': baseline_metrics['CE'] - feature_metrics['CE'], 'KL': feature_metrics['KL'], 'JS': feature_metrics['JS'], 'MAE': feature_metrics['MAE'], 'RMSE': feature_metrics['RMSE'], 'delta_CE_t': baseline_metrics['CE_t'] - feature_metrics['CE_t']}

Ajusta beta y el escalado únicamente con entrenamiento. Por defecto evalúa en validación; la opción test se reserva para una variable elegida previamente. Los splits suministrados deben purgar al menos el horizonte solicitado.

In [ ]:
if EJECUTAR_EXPERIMENTOS:
    train = df_modelo.iloc[SPLITS['train']].copy()
    disponibles = [f for f in STRICT_FEATURES if np.isfinite(load_feature(train, assets_modelo, f).to_numpy()).any()]
    QUALITY_TABLE = pd.DataFrame([audit_node_feature(train, f, assets_modelo, cfg_modelo) for f in STRICT_FEATURES])
    SYSTEM_QUALITY = audit_system_features(train, SYSTEM_FEATURES)
    AUDIT_PANEL = sampled_node_panel(train, assets_modelo, disponibles, cfg_modelo)
    REDUNDANCY = redundancy_report(AUDIT_PANEL, disponibles)
    CIRCULARITY = circularity_report(AUDIT_PANEL, disponibles)
    STABILITY = temporal_stability(train, assets_modelo, disponibles, cfg_modelo)
    STATIONARITY = stationarity_screen(train, assets_modelo, disponibles, cfg_modelo)
    PREDICTIVE_RESULTS_RAW = [screen_single_feature(df_modelo, assets_modelo, f, cfg_modelo, SPLITS) for f in disponibles]
    PREDICTIVE_TABLE = pd.DataFrame([{k:v for k,v in r.items() if k != 'delta_CE_t'} for r in PREDICTIVE_RESULTS_RAW]).sort_values('delta_CE', ascending=False)
    BOOTSTRAP_TABLE = pd.DataFrame([{'feature':r['feature'], **moving_block_bootstrap(r['delta_CE_t'], block_size=7)} for r in PREDICTIVE_RESULTS_RAW])
    TOP_FEATURES = PREDICTIVE_TABLE.head(10)['feature'].tolist()
    HALF_LIFE_TABLE = pd.DataFrame([{k:v for k,v in screen_single_feature(df_modelo, assets_modelo, f, cfg_modelo, SPLITS, horizon=h).items() if k != 'delta_CE_t'} for f in TOP_FEATURES for h in HORIZONTES])
    display(PREDICTIVE_TABLE)

Ejecuta, si se activa, las auditorías sobre train y la comparación predictiva sobre val. Elegir variables u horizontes según estas tablas no consulta el test. Los intervalos bootstrap son exploratorios y no corrigen por búsqueda múltiple.

In [ ]:
def run_xgb_family(family, df, assets, cfg, use_system_context=True):
    from xgboost import XGBRegressor
    features = MODEL_FAMILIES[family]
    system = SYSTEM_FEATURES if use_system_context else None
    X_train, y_train, _, _ = build_family_panel(df, assets, XGB_ROWS['train'], features, system, cfg)
    X_val, y_val, _, _ = build_family_panel(df, assets, XGB_ROWS['val'], features, system, cfg)
    X_test, y_test, meta_test, valid_test = build_family_panel(df, assets, XGB_ROWS['val'], features, system, cfg)
    model = XGBRegressor(objective='reg:squarederror', n_estimators=1200, learning_rate=0.03, max_depth=6, min_child_weight=20, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.05, reg_lambda=1.0, tree_method='hist', n_jobs=-1, random_state=42)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = model.predict(X_test)
    metrics = evaluate_family_delta(pred, meta_test, valid_test, cfg.eps)
    metrics.update({'family': family, 'model': 'XGBoost', 'n_features': len(features), 'evaluation': 'val'})
    return (model, metrics)

Conserva el modelo y la familia original. Importa su biblioteca solo al invocar la función y compara familias en validación, sin evaluar el test durante la selección.

In [ ]:
def run_lgb_family(family, df, assets, cfg, use_system_context=True):
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    features = MODEL_FAMILIES[family]
    system = SYSTEM_FEATURES if use_system_context else None
    X_train, y_train, _, _ = build_family_panel(df, assets, XGB_ROWS['train'], features, system, cfg)
    X_val, y_val, _, _ = build_family_panel(df, assets, XGB_ROWS['val'], features, system, cfg)
    X_test, y_test, meta_test, valid_test = build_family_panel(df, assets, XGB_ROWS['val'], features, system, cfg)
    model = LGBMRegressor(objective='regression', n_estimators=2000, learning_rate=0.025, num_leaves=63, max_depth=-1, min_child_samples=100, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.05, reg_lambda=1.0, n_jobs=-1, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[early_stopping(75, verbose=False), log_evaluation(0)])
    pred = model.predict(X_test)
    metrics = evaluate_family_delta(pred, meta_test, valid_test, cfg.eps)
    metrics.update({'family': family, 'model': 'LightGBM', 'n_features': len(features), 'evaluation': 'val'})
    return (model, metrics)

Conserva el modelo y la familia original. Importa su biblioteca solo al invocar la función y compara familias en validación, sin evaluar el test durante la selección.

In [ ]:
if EJECUTAR_EXPERIMENTOS:
    XGB_ROWS = {'train': SPLITS['train'], 'val': SPLITS['val']}
    XGB_MODELS, LGB_MODELS, resultados = {}, {}, []
    for familia in MODEL_FAMILIES:
        modelo, metricas = run_xgb_family(familia, df_modelo, assets_modelo, cfg_modelo)
        XGB_MODELS[familia] = modelo
        resultados.append(metricas)
    try:
        import lightgbm
    except ImportError:
        print('LightGBM no está instalado; sus comparaciones se omiten.')
    else:
        for familia in MODEL_FAMILIES:
            modelo, metricas = run_lgb_family(familia, df_modelo, assets_modelo, cfg_modelo)
            LGB_MODELS[familia] = modelo
            resultados.append(metricas)
    FAMILY_COMPARISON = pd.DataFrame(resultados).sort_values('delta_CE', ascending=False)
    display(FAMILY_COMPARISON)

def evaluar_modelo_final(modelo, familia):
    X, y, meta, valid = build_family_panel(df_modelo, assets_modelo, SPLITS['test'], MODEL_FAMILIES[familia], SYSTEM_FEATURES, cfg_modelo)
    return evaluate_family_delta(modelo.predict(X), meta, valid, cfg_modelo.eps)

Compara familias con validación. evaluar_modelo_final queda disponible para el modelo y la familia ya elegidos; no se invoca automáticamente ni se usa su resultado para volver a seleccionar.

## Auditoría causal opcional

In [ ]:
def reconstruir_original(raw, assets, cfg):
    df = raw.copy()
    n = len(df)
    sys_active = np.zeros(n)
    sys_trades = np.zeros(n)
    sys_buy = np.zeros(n)
    sys_sell = np.zeros(n)
    for chunk in chunks(assets, cfg.asset_block_size):
        O = load_base(df, chunk, 'O', cfg)
        observed = O > 0
        A = load_base(df, chunk, 'A', cfg)
        C = load_base(df, chunk, 'C', cfg).where(observed)
        B = load_base(df, chunk, 'B', cfg).where(observed)
        S = load_base(df, chunk, 'S', cfg).where(observed)
        sys_active += A.sum(axis=1).to_numpy()
        sys_trades += C.sum(axis=1).to_numpy()
        sys_buy += B.sum(axis=1).to_numpy()
        sys_sell += S.sum(axis=1).to_numpy()
    sys_flow = sys_buy + sys_sell
    SYS_imbalance = (sys_buy - sys_sell) / (sys_flow + cfg.eps)
    YS = df[cfg.ys_col].astype('float64')
    YS_log = np.log(YS.clip(lower=cfg.eps))
    YS_growth = YS_log.diff()
    YS_volatility = YS_growth.rolling(cfg.primary_window, min_periods=cfg.min_periods).std()
    system_features = pd.DataFrame({'SYS_active_nodes': sys_active, 'SYS_trades': sys_trades, 'SYS_imbalance': SYS_imbalance, 'YS_log': YS_log, 'YS_growth': YS_growth, 'YS_volatility': YS_volatility}, index=df.index).astype(cfg.dtype)
    df = merge_blocks(df, [system_features])
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        O = load_base(df, chunk, 'O', cfg)
        A = load_base(df, chunk, 'A', cfg)
        observed = O > 0
        P = load_base(df, chunk, 'P', cfg).where(observed)
        V = load_base(df, chunk, 'V', cfg).where(observed)
        logP = np.log(P.where(P > 0))
        ret = logP.diff(cfg.return_lag)
        volatility = ret.rolling(cfg.primary_window, min_periods=cfg.min_periods).std()
        V_valid = V.where((V > 0) & (A > 0))
        dlogV = np.log(V_valid).diff()
        volume_dispersion = rolling_robust_dispersion(dlogV, cfg.primary_window, cfg.min_periods, cfg.eps)
        VSTAB = np.exp(-volume_dispersion)
        blocks.extend([name_block(ret, 'return_k1', cfg), name_block(volatility, 'rolling_volatility', cfg), name_block(dlogV, 'dlogV', cfg), name_block(VSTAB, 'VSTAB', cfg)])
    df = merge_blocks(df, blocks)
    blocks = []
    SYS_trades = df['SYS_trades'].replace(0, np.nan)
    for chunk in chunks(assets, cfg.asset_block_size):
        O = load_base(df, chunk, 'O', cfg)
        A = load_base(df, chunk, 'A', cfg)
        C = load_base(df, chunk, 'C', cfg)
        observed = O > 0
        C = C.where(observed)
        A_masked = A.where(observed)
        logC = np.log1p(C.clip(lower=0))
        TI = robust_z_causal(logC, cfg.primary_window, cfg.min_periods, cfg.eps)
        PERS_C = rolling_autocorr_lag1(logC, cfg.primary_window, cfg.min_periods)
        Arel = C.div(SYS_trades, axis=0)
        CA = A_masked.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean()
        CO = O.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean()
        blocks.extend([name_block(TI, 'TI', cfg), name_block(PERS_C, 'PERS_C', cfg), name_block(Arel, 'Arel', cfg), name_block(CA, 'CA', cfg), name_block(CO, 'CO', cfg)])
    df = merge_blocks(df, blocks)
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        O = load_base(df, chunk, 'O', cfg)
        observed = O > 0
        B = load_base(df, chunk, 'B', cfg).where(observed)
        S = load_base(df, chunk, 'S', cfg).where(observed)
        logS1 = np.log1p(S.clip(lower=0))
        SI = robust_z_causal(logS1, cfg.primary_window, cfg.min_periods, cfg.eps)
        SG = np.log(S.clip(lower=0) + cfg.eps).diff()
        SP = SI.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean()
        total = B + S
        imbalance = (B - S) / (total + cfg.eps)
        sell_share = S / (total + cfg.eps)
        LR_BS = np.log((B + cfg.eps) / (S + cfg.eps))
        BIL = 2.0 * np.minimum(B, S) / (total + cfg.eps)
        blocks.extend([name_block(SI, 'SI', cfg), name_block(SG, 'SG', cfg), name_block(SP, 'SP', cfg), name_block(imbalance, 'imbalance', cfg), name_block(sell_share, 'sell_share', cfg), name_block(LR_BS, 'LR_BS', cfg), name_block(BIL, 'BIL', cfg)])
    df = merge_blocks(df, blocks)
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        O = load_base(df, chunk, 'O', cfg)
        observed = O > 0
        C = load_base(df, chunk, 'C', cfg).where(observed)
        S = load_base(df, chunk, 'S', cfg).where(observed)
        R = load_base(df, chunk, 'R', cfg).where(observed)
        L = load_base(df, chunk, 'L', cfg).where(observed)
        BIL = load_feature(df, chunk, 'BIL')
        BIL_disp = rolling_robust_dispersion(BIL.diff(), cfg.primary_window, cfg.min_periods, cfg.eps)
        BILST = BIL.ewm(span=cfg.ewm_span, adjust=False, min_periods=cfg.min_periods).mean() * np.exp(-BIL_disp)
        logC = np.log1p(C.clip(lower=0))
        logS = np.log1p(S.clip(lower=0))
        logL = np.log1p(L.clip(lower=0))
        ABS = logC / (R + cfg.eps)
        ABS_S = logS / (R + cfg.eps)
        RLIQ = logC / (1.0 + logL)
        RLIQ_S = logS / (1.0 + logL)
        LIQ = -logL
        blocks.extend([name_block(BILST, 'BILST', cfg), name_block(ABS, 'ABS', cfg), name_block(ABS_S, 'ABS_S', cfg), name_block(RLIQ, 'RLIQ', cfg), name_block(RLIQ_S, 'RLIQ_S', cfg), name_block(LIQ, 'LIQ', cfg)])
    df = merge_blocks(df, blocks)
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        O = load_base(df, chunk, 'O', cfg)
        observed = O > 0
        C = load_base(df, chunk, 'C', cfg).where(observed)
        R = load_base(df, chunk, 'R', cfg).where(observed)
        L = load_base(df, chunk, 'L', cfg).where(observed)
        B = load_base(df, chunk, 'B', cfg).where(observed)
        S = load_base(df, chunk, 'S', cfg).where(observed)
        ret = load_feature(df, chunk, 'return_k1')
        imbalance = load_feature(df, chunk, 'imbalance')
        logC = np.log1p(C.clip(lower=0))
        pressure = imbalance.abs() * logC
        beta = rolling_beta_with_intercept(pressure, ret.abs(), cfg.primary_window, cfg.min_periods, cfg.eps)
        IR = 1.0 / (beta + cfg.eps)
        zR = robust_z_causal(R, cfg.primary_window, cfg.min_periods, cfg.eps)
        zL = robust_z_causal(L, cfg.primary_window, cfg.min_periods, cfg.eps)
        zRet = robust_z_causal(ret.abs(), cfg.primary_window, cfg.min_periods, cfg.eps)
        STRESS = zR + zL + zRet
        threshold = rolling_stress_threshold(STRESS, cfg.primary_window, cfg.min_periods, cfg.stress_quantile)
        SSR = conditional_rolling_median_ratio(C, STRESS, threshold, cfg.primary_window, cfg.min_periods, cfg.eps)
        SSR_S = conditional_rolling_median_ratio(S, STRESS, threshold, cfg.primary_window, cfg.min_periods, cfg.eps)
        AST_S = S / (C + cfg.eps)
        blocks.extend([name_block(IR, 'IR', cfg), name_block(STRESS, 'STRESS', cfg), name_block(SSR, 'SSR', cfg), name_block(SSR_S, 'SSR_S', cfg), name_block(AST_S, 'AST_S', cfg)])
    df = merge_blocks(df, blocks)
    gp0_inputs = ('TI', 'CA', 'VSTAB', 'PERS_C', 'BIL', 'ABS', 'RLIQ', 'IR', 'SSR')
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        components = [load_feature(df, chunk, feature) for feature in gp0_inputs]
        gp0 = composite_positive_proxy(components, cfg)
        STRESS = load_feature(df, chunk, 'STRESS')
        threshold = rolling_stress_threshold(STRESS, cfg.primary_window, cfg.min_periods, cfg.stress_quantile)
        baseline = gp0.shift(1).rolling(cfg.primary_window, min_periods=cfg.min_periods).median()
        REC = recovery_speed_causal(gp0=gp0, stress=STRESS, stress_threshold=threshold, baseline=baseline, tolerance=cfg.recovery_tolerance, hold_bars=cfg.recovery_hold_bars, alpha=cfg.recovery_event_ewm_alpha, eps=cfg.eps)
        gp = composite_positive_proxy(components + [REC], cfg)
        chunk_blocks = [name_block(gp0, 'gp0', cfg), name_block(REC, 'REC', cfg), name_block(gp, 'gp', cfg)]
        blocks.extend(chunk_blocks)
    df = merge_blocks(df, blocks)
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        gp = load_feature(df, chunk, 'gp')
        G = {}
        chunk_blocks = []
        for w in cfg.multiscale_windows:
            G[w] = gp.ewm(span=w, adjust=False, min_periods=max(2, min(w, cfg.min_periods))).mean()
            chunk_blocks.append(name_block(G[w], f'gp_W{w}', cfg))
        sum_log = None
        for w in cfg.multiscale_windows:
            lg = np.log(G[w].clip(lower=cfg.eps))
            sum_log = lg if sum_log is None else sum_log + lg
        fgp_nested = np.exp(sum_log / len(cfg.multiscale_windows))
        chunk_blocks.append(name_block(fgp_nested, 'fgp_nested', cfg))
        blocks.extend(chunk_blocks)
    df = merge_blocks(df, blocks)
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        gp = load_feature(df, chunk, 'gp')
        dlog_gp = np.log(gp.clip(lower=cfg.eps)).diff()
        gp_dispersion = rolling_robust_dispersion(dlog_gp, cfg.primary_window, cfg.min_periods, cfg.eps)
        NSTAB = np.exp(-gp_dispersion)
        NPERS = rolling_autocorr_lag1(gp, cfg.primary_window, cfg.min_periods)
        logs = []
        for w in cfg.multiscale_windows:
            G = load_feature(df, chunk, f'gp_W{w}')
            logs.append(np.log(G.clip(lower=cfg.eps)))
        stack = np.stack([x.to_numpy() for x in logs], axis=2)
        variance = np.nanvar(stack, axis=2)
        NCOH = pd.DataFrame(np.exp(-variance), index=df.index, columns=chunk)
        STRESS = load_feature(df, chunk, 'STRESS')
        threshold = rolling_stress_threshold(STRESS, cfg.primary_window, cfg.min_periods, cfg.stress_quantile)
        NRET = conditional_rolling_median_ratio(gp, STRESS, threshold, cfg.primary_window, cfg.min_periods, cfg.eps)
        chunk_blocks = [name_block(NSTAB, 'NSTAB', cfg), name_block(NPERS, 'NPERS', cfg), name_block(NCOH, 'NCOH', cfg), name_block(NRET, 'NRET', cfg)]
        blocks.extend(chunk_blocks)
    df = merge_blocks(df, blocks)
    A_all = load_base(df, assets, 'A', cfg)
    gp_active = load_feature(df, assets, 'gp').where(A_all > 0)
    NRANK = np.exp(-gp_active.rank(axis=1, pct=True, method='average').diff().abs())
    blocks = [name_block(NRANK, 'NRANK', cfg)]
    q_logs = []
    for w in cfg.multiscale_windows:
        G = load_feature(df, assets, f'gp_W{w}').where(A_all > 0)
        total = G.sum(axis=1, min_count=1).replace(0, np.nan)
        qg = G.div(total, axis=0)
        blocks.append(name_block(qg, f'qg_W{w}', cfg))
        q_logs.append(np.log(qg.clip(lower=cfg.eps)).to_numpy())
    stack = np.stack(q_logs, axis=2)
    NEST = pd.DataFrame(np.exp(-np.nanvar(stack, axis=2)), index=df.index, columns=assets)
    blocks.append(name_block(NEST, 'NEST', cfg))
    df = merge_blocks(df, blocks)
    N_inputs = ('NSTAB', 'NPERS', 'NCOH', 'NRET', 'NRANK', 'NEST')
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        components = [load_feature(df, chunk, feature) for feature in N_inputs]
        N_context = composite_positive_proxy(components, cfg)
        fgp = load_feature(df, chunk, 'fgp_nested')
        yp = load_base(df, chunk, 'yp', cfg)
        yp_safe = yp.clip(lower=cfg.eps, upper=1.0)
        surprise = -np.log(yp_safe)
        N_theory = fgp / (surprise + cfg.eps)
        gp = load_feature(df, chunk, 'gp')
        E_context = gp / (N_context + cfg.eps)
        E_theory = gp / (N_theory + cfg.eps)
        chunk_blocks = [name_block(N_context, 'N_context', cfg), name_block(N_theory, 'N_theory', cfg), name_block(E_context, 'E_context', cfg), name_block(E_theory, 'E_theory', cfg)]
        blocks.extend(chunk_blocks)
    df = merge_blocks(df, blocks)
    E_all = load_feature(df, assets, 'E_context').astype('float32')
    A_all = load_base(df, assets, 'A', cfg).astype('float32')
    E_active = E_all.where(A_all > 0)
    E_median = E_active.median(axis=1)
    E_rank = E_active.rank(axis=1, pct=True, method='average')
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        E = E_all.loc[:, chunk]
        logE = np.log(E.clip(lower=cfg.eps))
        Erel = E.sub(E_median, axis=0)
        dE = E.diff()
        ddE = dE.diff()
        ESH = robust_z_causal(E, cfg.primary_window, cfg.min_periods, cfg.eps)
        ERANK = E_rank.loc[:, chunk]
        chunk_blocks = [name_block(logE, 'Ectx_logE', cfg), name_block(Erel, 'Ectx_Erel', cfg), name_block(dE, 'Ectx_dE', cfg), name_block(ddE, 'Ectx_ddE', cfg), name_block(ESH, 'Ectx_ESH', cfg), name_block(ERANK, 'Ectx_ERANK', cfg)]
        blocks.extend(chunk_blocks)
    df = merge_blocks(df, blocks)
    E_all = load_feature(df, assets, 'E_theory').astype('float32')
    A_all = load_base(df, assets, 'A', cfg).astype('float32')
    E_active = E_all.where(A_all > 0)
    E_median = E_active.median(axis=1)
    E_rank = E_active.rank(axis=1, pct=True, method='average')
    blocks = []
    for chunk in chunks(assets, cfg.asset_block_size):
        E = E_all.loc[:, chunk]
        logE = np.log(E.clip(lower=cfg.eps))
        Erel = E.sub(E_median, axis=0)
        dE = E.diff()
        ddE = dE.diff()
        ESH = robust_z_causal(E, cfg.primary_window, cfg.min_periods, cfg.eps)
        ERANK = E_rank.loc[:, chunk]
        chunk_blocks = [name_block(logE, 'Eth_logE', cfg), name_block(Erel, 'Eth_Erel', cfg), name_block(dE, 'Eth_dE', cfg), name_block(ddE, 'Eth_ddE', cfg), name_block(ESH, 'Eth_ESH', cfg), name_block(ERANK, 'Eth_ERANK', cfg)]
        blocks.extend(chunk_blocks)
    df = merge_blocks(df, blocks)
    return df

Agrupa las mismas 13 etapas en una función para verificar invariancia por prefijos. No incluye lecturas, escrituras, proyección de puestos ni objetivos.

In [ ]:
def verificar_prefijos(raw, assets, cfg, cortes):
    completo = reconstruir_original(raw, assets, cfg)
    resultados = []
    for corte in cortes:
        prefijo = reconstruir_original(raw.iloc[:corte].copy(), assets, cfg)
        pd.testing.assert_frame_equal(completo.iloc[:corte], prefijo, check_exact=False, rtol=1e-6, atol=1e-7)
        resultados.append({'filas': corte, 'invariante': True})
    return pd.DataFrame(resultados)

EJECUTAR_AUDITORIA_CAUSAL = False
if EJECUTAR_AUDITORIA_CAUSAL:
    raw = df_original.loc[:, ['bar_end', cfg.ys_col, *requeridas]].copy()
    display(verificar_prefijos(raw, assets_originales, cfg, [len(raw)//2, 3*len(raw)//4]))

Cuando se active, comprueba que añadir futuro no cambia ninguna fila pasada de las variables originales. La proyección utiliza únicamente rankings desplazados un día. Esta auditoría no sustituye la revisión de disponibilidad de los datos de origen ni de la composición histórica del universo.